In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.4 GQA / MQA

At inference the KV cache dominates memory. **Grouped-query attention** shares one
set of key/value heads across several query heads; **multi-query** is the extreme
of a single KV head. Same query expressivity, a fraction of the KV memory.

In [ ]:
from pocketlm import PocketLM, PocketLMConfig

def kv_params(n_kv):
    cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4,
                         n_embd=32, n_kv_head=n_kv)
    attn = PocketLM(cfg).blocks[0].attn
    return sum(p.numel() for p in attn.c_kv.parameters())

print("full (4 KV heads):", kv_params(4))
print("GQA  (2 KV heads):", kv_params(2))
print("MQA  (1 KV head) :", kv_params(1))

Halving the KV heads halves the KV projection (and the cache it feeds). The model
still works, the no-future-leak invariant still holds, the memory bill drops.

In [ ]:
assert kv_params(1) < kv_params(2) < kv_params(4)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4, n_embd=32,
                     n_kv_head=1)
logits, _ = PocketLM(cfg)(torch.randint(0, 20, (1, 8)))
assert logits.shape == (1, 8, 20)